In [ ]:
%pip install -e ../../../
%pip install scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from utils import upload_csv_files_glob, reconstruct_CBS_fullname
from water_futures_battle.core.base_model import DynamicProperties

In [ ]:
static_properties_path = "../jurisdictions-static_properties.xlsx"
municipalities_df = pd.read_excel(static_properties_path,
                                  sheet_name='municipalities')

municipalites_dynamic_properties = DynamicProperties('municipalities-dynamic_properties')

if not Path('../raw-data/CBS-regionale_kerncijfers_nederland').exists():
    with zipfile.ZipFile('../raw-data/CBS-regionale_kerncijfers_nederland.zip', 'r') as zip_ref:
        zip_ref.extractall('../raw-data')

In [ ]:
# Upload the population files in a single dataframe with these columns:
POPULATION_BY_URBAN_CLASS_COLUMNS = [
    'CBS_fullname', 'year',
    'population', 'population-very_strong_urban',
    'population-strong_urban', 'population-moderate_urban',
    'population-little_urban', 'population-non_urban'
]
POPULATION_BY_URBAN_CLASS_DTYPES = [
    str, int
] + ['Int64']*6 

pop_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-population_by_urban_class",
    columns_names=POPULATION_BY_URBAN_CLASS_COLUMNS,
    columns_dtypes=POPULATION_BY_URBAN_CLASS_DTYPES
)

# Match every municipality with the 'CBS_fullname' i.e., the name with 
municipalities_df['CBS_fullname'] = municipalities_df.apply(
    lambda x: reconstruct_CBS_fullname(x, cbs_df=pop_df, expected_matches=25), axis=1
    )

pop_df = pd.merge(pop_df, municipalities_df[['CBS_fullname', 'cbs_id']], on='CBS_fullname', how='inner')

pop_df['timestamp'] = pop_df['year'].apply(lambda x: pd.to_datetime(str(x)))
pop_df.sort_index(inplace=True)


# We decided that we don't need population by urbanity_class anymore so we will drop them
# and kep only the total population.
dyn_df = pop_df.pivot(index='timestamp', columns='cbs_id', values='population')

pop_nl_df = pd.read_csv(
    '../raw-data/population-projections.csv',
    index_col=0,
    parse_dates=True
)
# Columns are observation, fcst 2023, fcst 2024, confidence interval 67, confidence interval 95
# We move the fcst 2024 in the future of the observation (column NL0000) 
Y2025 = pd.to_datetime("2025-01-01")
future_mask = pop_nl_df.index >= Y2025
pop_nl_df['NL0000'] = pop_nl_df['Measured'].where(~future_mask).astype(float)*1e6
pop_nl_df['NL0000-0.5'] = pop_nl_df['Forecast 2024-2070'].astype(float)*1e6
pop_nl_df[['NL0000-0.025', 'NL0000-0.975']] = pop_nl_df['Confidence interval (95%)'].apply(
    lambda x: pd.Series(x.split('–')) if pd.notnull(x) else pd.Series([None, None])
).astype(float)*1e6 # Careful with the simbol because it is not the normal dash
pop_nl_df[['NL0000-0.165', 'NL0000-0.835']] = pop_nl_df['Confidence interval (67%)'].apply(
    lambda x: pd.Series(x.split('–')) if pd.notnull(x) else pd.Series([None, None])
).astype(float)*1e6

full_pop_df = dyn_df.copy()
full_pop_df['NL0000'] = full_pop_df.sum(axis=1, skipna=True)
# display(full_pop_df['NL0000']-pop_nl_df['NL0000'])

full_pop_df = pd.concat([
    full_pop_df,
    pop_nl_df.loc[future_mask, ['NL0000-0.025','NL0000-0.165','NL0000-0.5','NL0000-0.835','NL0000-0.975']].astype('Int64')
    ],
    axis=1
)
full_pop_df.index.name = 'timestamp'

municipalites_dynamic_properties['population'] = full_pop_df.copy()

### Helping information from population

#### Population at some points in time

This helps me tuning other modules.

- National population for pop vs km

- Province poulation for leak

In [ ]:
display(full_pop_df['NL0000'])

#### Population to pipes

Now that we have the population every year and that we have the length of pipes in the NL from the Vewin report (Figure 3.6) we can get the ratio.

In [ ]:
import numpy as np
pop_vs_km = [
    [10093554, 30000],#0,0274473925
    [11486631, 40000],#0,0034825004
    [12294732, 58000],#0,0047177485
    [13038526, 65000],#0,0049854272
    [13666335, 78000],#0,0057075955
    [14149800, 81000],#0,0057247862
    [14491632, 85000],#0,0058657098
    [14951510, 90000],
    [15459006, 100000],
    [15925513, 105000],
    [16319868, 115000],
    [16615394, 119000],
    [16939923, 120000],
    [17441500, 121000]]

# Average number of kilometers per 10k population
km_per_10k = float(np.mean([(km / pop) * 10000 for pop, km in pop_vs_km]))
display(km_per_10k)

In [ ]:
pvs = municipalities_df['province'].unique()
for pv_id in pvs:
    municipalities_in_pv = municipalities_df.loc[municipalities_df['province'] == pv_id, 'cbs_id']
    dyn_df[pv_id] = dyn_df[municipalities_in_pv].sum(axis=1, skipna=True)

display(dyn_df[pvs])
# Get the approx cost to apply pop * 57.7 (i.e.  length km)
display(dyn_df[pvs]*57.7e-4*450)

In [ ]:
# Environment and land use files 
ELU_COLUMNS = [
    'CBS_fullname', 'year',
    'surface-land', 'surface-water-inland', 'surface-water-open'
]
ELU_DTYPES = [
    str, int
] + [float]*3
elu_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-elu",
    columns_names=ELU_COLUMNS,
    columns_dtypes=ELU_DTYPES
)

elu_df = pd.merge(elu_df, municipalities_df[['CBS_fullname', 'cbs_id']], on='CBS_fullname', how='inner')

elu_df['timestamp'] = elu_df['year'].apply(lambda x: pd.to_datetime(str(x)))
elu_df.sort_index(inplace=True)

for sheet_name in ELU_COLUMNS[2:]:
    #elu_df[sheet_name] = elu_df[sheet_name].astype(str).str.replace(',', '.').astype(float)

    dyn_df = elu_df.pivot(index='timestamp', columns='cbs_id', values=sheet_name)

    municipalites_dynamic_properties[sheet_name] = dyn_df

In [ ]:
# Buildings (number)
BUILDINGS_COLUMNS = [
    'CBS_fullname', 'year',
    'n_houses'
]
BUILDINGS_DTYPES = [
    str, int,
    'Int64'
]
buildings_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-num_buildings",
    columns_names=BUILDINGS_COLUMNS,
    columns_dtypes=BUILDINGS_DTYPES
)
buildings_df = pd.merge(buildings_df, municipalities_df[['CBS_fullname', 'cbs_id']], on='CBS_fullname', how='inner')

buildings_df['timestamp'] = buildings_df['year'].apply(lambda x: pd.to_datetime(str(x)))
buildings_df.sort_index(inplace=True)

for sheet_name in BUILDINGS_COLUMNS[2:]:
    #buildings_df[sheet_name] = buildings_df[sheet_name].astype(str).str.replace(',', '.').astype(float)

    dyn_df = buildings_df.pivot(index='timestamp', columns='cbs_id', values=sheet_name)

    municipalites_dynamic_properties[sheet_name] = dyn_df

## Number of Businesses

We downloaded the number of businesses by activity and by municipality.

We immediately realized that since we don't have enough data to have a water 
demand profile for each business activity, we needed to use the aggregated value
(number of businesses by municipality).

Moreover, the data are available only between 2015 and 2025 (included).

So, given that the number of businesses has more or less a linear trend, the first
approach was to extrapolate/reconstruct the series from 2000 to 2014 using a linear
regression per each municipality data (function `reconstruct_until_2000_LinReg`).

After doing this, we used the aggregated data at national level and the values in
"development of drinking water sales to the business market" (Figure 3.9 in Vewin, 2022),
to reconstruct the "per business demand".

However, the "per business demand" for the early years (2000-2005) was extremely
high both in absolute value (2000 L/business/day) and in relative terms to the 
"per business demand" after 2015 (4x).

Thus, after this analysis from the Ferrara team, we decided to use a custom made
trend (see variable `ratio_to_2015_NoB` and function `reconstruct_until_2000_UniFe`)
by looking at the number of businesses in the NL [../raw-data/CBS_how_many_companies_2021.html]
and with some manual fine tuning.

The same trend is used for all municipalities.

In [ ]:
# Businesses by activity

# We have data from 2013. Let's do a simple regression and fill the previous years.
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12802)
from sklearn.linear_model import LinearRegression

BUSINESSES_COLUMNS = [
    'CBS_fullname', 'year',
]+[f"n_businesses-BT{i}" for i in range(1,9)]
BUSINESS_DTYPES = [
    str, int
]+['Int64']*8
businesses_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-businesses_by_activity",
    columns_names=BUSINESSES_COLUMNS,
    columns_dtypes=BUSINESS_DTYPES
)
businesses_df = pd.merge(businesses_df, municipalities_df[['CBS_fullname', 'cbs_id']], on='CBS_fullname', how='inner')

businesses_df['timestamp'] = businesses_df['year'].apply(lambda x: pd.to_datetime(str(x)))
businesses_df.sort_index(inplace=True)

# let's add the total number of business (we could have donwloaded it but too late)
businesses_df['n_businesses'] = businesses_df[BUSINESSES_COLUMNS[2:]].sum(axis=1, skipna=False)

def reconstruct_until_2000_LinReg(df_original: pd.DataFrame):
    new_index = pd.date_range(
        start='2000-01-01',
        end=df_original.index.min()-pd.Timedelta(days=1),
        freq='YS'
    ) # Year Start frequency

    df_reconstructed = pd.DataFrame(index=new_index, columns=df_original.columns)
    df_combined = pd.concat([df_reconstructed, df_original])
    df_combined.index.name = df_original.index.name
    # Convert dates to a numerical format (number of days since the earliest date)
    # This is the feature (X) used for the linear regression model.
    df_combined['time_num'] = (df_combined.index - df_combined.index.min()).days.values.reshape(-1, 1)

    df_final_reconstructed = df_combined.copy()
    for col in df_original.columns:
        muni = col

        begin_date = municipalities_df.set_index('cbs_id').loc[muni, 'begin_date'] 

        # 1. Prepare data for model training (only use the original non-NaN data points)
        df_train = df_combined.dropna(subset=[col])

        if df_train.empty: # Municipality closed before 2013
            end_date = municipalities_df.set_index('cbs_id').loc[muni, 'end_date']
            fill_values = pd.Series(np.nan, index=df_reconstructed.index)
            fill_values.loc[fill_values.index < end_date] = 0
            
            predictions = fill_values.values

        else:
            X_train = df_train['time_num'].values.reshape(-1, 1)
            y_train = df_train[col].values

            # 2. Train the Linear Regression Model
            model = LinearRegression()
            model.fit(X_train, y_train)

            # 3. Predict the missing values (the pre-2013 rows)
            df_predict = df_combined.loc[df_reconstructed.index]
            X_predict = df_predict['time_num'].values.reshape(-1, 1)

            predictions = model.predict(X_predict)
            predictions = np.round(predictions)
            predictions[predictions < 0] = 0

        to_reconstruct_dates = df_reconstructed.index[df_reconstructed.index >= begin_date]
        df_final_reconstructed.loc[to_reconstruct_dates, col] = predictions[df_reconstructed.index >= begin_date]

    df_final_reconstructed.drop(columns='time_num', inplace=True)

    return df_final_reconstructed

def reconstruct_until_2000_UniFe(df_original: pd.DataFrame):
    """
    After seeing the results of the reconstruct_until_2000_LinReg function, we realized
    that the trend goes down to quickly and to match the Vewin report we need 
    extremely high initial per business user demand.
    Thus, by looking at the number of businesses in the NL [../raw-data/CBS_how_many_companies_2021.html]
    and a bit manually we reconstructed the total number of businesses at national level
    """
    munis_ids = df_original.columns
    n_businesses_NL = df_original.sum(axis=1, skipna=True)

    # Ratio between the number of businesses in the NL in a given year we need
    # to reconstruct with respect to the 2015 value
    ratio_to_2015_NoB = np.array([ 
            0.70051,
            0.71574,
            0.73131,
            0.74721,
            0.76345,
            0.78005,
            0.79701,
            0.81434,
            0.83204,
            0.85013,
            0.86862,
            0.88750,
            0.90680,
            0.92651,
            0.94666
        ])
    YoY_ratio_NoB = ratio_to_2015_NoB[:-1]/ratio_to_2015_NoB[1:]
    YoY_ratio_NoB = np.concatenate([np.array([np.mean(YoY_ratio_NoB)]), YoY_ratio_NoB]) # Simply readd a value between 1999/2000

    years_to_reconstruct=pd.date_range(
        start='2000-01-01',
        end='2014-12-31',
        freq='YS'
    )
    years_to_reconstruct_plus1 = years_to_reconstruct + pd.DateOffset(years=1)
    # the only thing remaining to do is split the municipalities that were open
    # between 2000 and 2014
    # We go backward splitting the businesses in the municipalities. We use the
    # same proportion of population data
    lookup_df = municipalities_df.set_index('cbs_id')
    for year, previous_year, ratio in zip(years_to_reconstruct_plus1[::-1], years_to_reconstruct[::-1], YoY_ratio_NoB[::-1]):
        df_original.loc[previous_year, :] = (ratio*df_original.loc[year, :]).round().astype('Int64')

        closing_munis = municipalities_df[municipalities_df['end_date'] == year.strftime('%Y-%m-%d')]

        dest_munis_map: dict[str, list[str]] = {}
        for idx, closing_muni in closing_munis.iterrows():
            cm_id = closing_muni['cbs_id']
            dest_munis_ids = str(lookup_df.loc[cm_id, 'destination_cbs_ids']).split(';')
            for dest_muni_id in dest_munis_ids:
                if dest_muni_id not in dest_munis_map:
                    dest_munis_map[dest_muni_id] = []    
                dest_munis_map[dest_muni_id].append(cm_id)

        closing_munis_businesses: dict[str, int] = {m: 0. for m in closing_munis['cbs_id']}
        for dest_muni_id, orig_munis_ids in dest_munis_map.items():
            # Option 1 (Dest muni is opening)
            # its value in previous year is nan and its splits its number of years equally
            if lookup_df.loc[dest_muni_id, "begin_date"] == year.strftime('%Y-%m-%d'):
                agg_value = int(df_original.loc[previous_year, dest_muni_id])
                df_original.loc[previous_year, dest_muni_id] = np.nan
                n_orig_munis = len(orig_munis_ids)
                if n_orig_munis == 0:
                    continue
                for m in orig_munis_ids:
                    closing_munis_businesses[m] += round((agg_value/n_orig_munis))
                
            else:
                pass

        for m, n_businesses in closing_munis_businesses.items():
            df_original.loc[previous_year, m] = n_businesses

    return df_original.sort_index()

def reconstruct_until_2000_UniFePlus(df_original: pd.DataFrame):
    """
    After seeing the UniFe approach, I noticed that their trend had a constant
    year on year ratio. So I used the year on year data at the national level
    from 2015 to 2025 to create the bounds for a uniform distribution from which
    to extract the value.
    """
    munis_ids = df_original.columns
    n_businesses_NL = df_original.sum(axis=1, skipna=True)

    # Ratio between the number of businesses in the NL in a given year we need
    # to reconstruct with respect to the 2015 value

    years_to_reconstruct=pd.date_range(
        start='2000-01-01',
        end='2014-12-31',
        freq='YS'
    )
    years_to_reconstruct_plus1 = years_to_reconstruct + pd.DateOffset(years=1)
    # the only thing remaining to do is split the municipalities that were open
    # between 2000 and 2014
    # We go backward splitting the businesses in the municipalities. We use the
    # same proportion of population data
    lookup_df = municipalities_df.set_index('cbs_id')
    for year, previous_year in zip(years_to_reconstruct_plus1[::-1], years_to_reconstruct[::-1]):
        df_original.loc[previous_year, :] = (
            RNG.uniform(0.975, 0.980)*df_original.loc[year, :]
            ).round().astype('Int64')

        closing_munis = municipalities_df[municipalities_df['end_date'] == year.strftime('%Y-%m-%d')]

        dest_munis_map: dict[str, list[str]] = {}
        for idx, closing_muni in closing_munis.iterrows():
            cm_id = closing_muni['cbs_id']
            dest_munis_ids = str(lookup_df.loc[cm_id, 'destination_cbs_ids']).split(';')
            for dest_muni_id in dest_munis_ids:
                if dest_muni_id not in dest_munis_map:
                    dest_munis_map[dest_muni_id] = []    
                dest_munis_map[dest_muni_id].append(cm_id)

        closing_munis_businesses: dict[str, int] = {m: 0. for m in closing_munis['cbs_id']}
        for dest_muni_id, orig_munis_ids in dest_munis_map.items():
            # Option 1 (Dest muni is opening)
            # its value in previous year is nan and its splits its number of years equally
            if lookup_df.loc[dest_muni_id, "begin_date"] == year.strftime('%Y-%m-%d'):
                agg_value = int(df_original.loc[previous_year, dest_muni_id])
                df_original.loc[previous_year, dest_muni_id] = np.nan
                n_orig_munis = len(orig_munis_ids)
                if n_orig_munis == 0:
                    continue
                for m in orig_munis_ids:
                    closing_munis_businesses[m] += round((agg_value/n_orig_munis))
                
            else:
                pass

        for m, n_businesses in closing_munis_businesses.items():
            df_original.loc[previous_year, m] = n_businesses

    return df_original.sort_index()

# I need to change 
for sheet_name in ['n_businesses']: # BUSINESSES_COLUMNS[2:]:
    dyn_df = businesses_df.pivot(index='timestamp', columns='cbs_id', values=sheet_name)
    dyn_df.dropna(axis=0, how='all', inplace=True)

    dyn_df = reconstruct_until_2000_UniFe(dyn_df)

    # dyn_dfpro = dyn_df.copy()
    # dyn_dfpro = dyn_dfpro.sum(axis=1)
    # y2y_df = dyn_dfpro.shift(1) / dyn_dfpro

    municipalites_dynamic_properties[sheet_name] = dyn_df

## Associated Water Pattern

Ferrara associated the water demand pattern to each municipality in the historical period.
We just need to do that for each municipality in their begin year

In [ ]:
association_df =  pd.read_csv(
    '../raw-data/Municipality_AssociatedCurves.csv',
    delimiter=';',
    header=1,
    index_col=0
)
# Ferrara prepared the file using the row order as the columns order of the other properties (populaiton, etc)
# [:605] to remove the extra NL0000 COLUMN added in the last version
ASSOC_COLUMNS = [m for m in municipalites_dynamic_properties['population'].columns.copy() if m.startswith('GM')]

for name, col in zip(["assoc_dem_pat-residential-1", "assoc_dem_pat-residential-2", "assoc_dem_pat-business"],
                     association_df.columns):
    data = {gm: f"DP{i:02d}" for gm, i in zip(ASSOC_COLUMNS, association_df[col].astype(int))}
    
    # Every column in df is in 2000, we need to copy the base row every year and
    # if it is open it goes there otherwise it goes ""
    df_years = []
    df_index = pd.date_range(start="2000-01-01", end="2024-12-31", freq='YS')
    for ts in df_index:
        begin_dates = pd.to_datetime(municipalities_df['begin_date'])
        end_dates = pd.to_datetime(municipalities_df['end_date'], errors='coerce')
        dates_mask = (begin_dates <= ts) & (pd.isna(end_dates) | (ts < end_dates))
        dates_mask.index = municipalities_df['cbs_id']

        data_year = {gm: data[gm] if dates_mask.loc[gm] else "" for gm in ASSOC_COLUMNS}
        df_years.append(data_year)

    df = pd.DataFrame(df_years)
    df.index = df_index
    df.index.name = 'timestamp'

    municipalites_dynamic_properties[name] = df.copy()

## Non Revenue Water Demand 

According to the Vewing 2022 report NRW was between 50 million m3/year in 2000 and 75 in 2020 (Figure 3.10).
From the same report, in the same years demand went from 1190 to 1220 million m3/year (Figure 3.7).
Therefore there was between 4 and 6 % loss.
In the same years there were 107000 and 120000 km of pipes (Figure 3.6 and Table 3.2)
Which means that in 2000 there were 1.281 m3/km/day (rounded up) and in 2020 1.92 m3/km/day
Way below our first class. If we put it in all of them in the first class it would be way higher.
Anyway.

In [ ]:
# NRW class and years in NRW class
Y2000 = pd.to_datetime("2000-01-01")
net_avg_age_df = pd.DataFrame(
    data=[RNG.integers(25, 45, size=len(municipalities_df))],
    columns=municipalities_df['cbs_id'],
    index=[Y2000]
)
net_avg_age_df = net_avg_age_df.astype(float)
net_avg_age_df.index.name = "timestamp"

# Municipalities that are not active in 2000 should disappear and the evaluator adds their value at creation
drop_nrw_munis = []
for idx, muni_data in municipalities_df.iterrows():
    if pd.to_datetime(muni_data['begin_date']) > Y2000:
        drop_nrw_munis.append(muni_data['cbs_id'])

net_avg_age_df.drop(columns=drop_nrw_munis, inplace=True)

for sheet_name, df in zip(['dist_network-age-avg'], [net_avg_age_df]):
    municipalites_dynamic_properties[sheet_name] = df

## Average Disposable Income

Downloaded from CBS like other properties.
Data before 2010 are missing .
We assume a randomized 2% variation in year on year

For year 2024, we use the preliminary figures published in https://www.cbs.nl/en-gb/figures/detail/83739eng

Here it says that at the national level, they estimate an increase of 4.5%.
Thus we generate a string between 4 and 5 of all muncipalities and make sure the average is 4.5%

In [ ]:
# Average disposable income files
ADI_COLUMNS = [
    'CBS_fullname', 'year',
    'disposable_income-avg'
]
ADI_DTYPES = [
    str, int,
    float
]
adi_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-disposable_income",
    columns_names=ADI_COLUMNS,
    columns_dtypes=ADI_DTYPES
)
adi_df = pd.merge(adi_df, municipalities_df[['CBS_fullname', 'cbs_id']], on='CBS_fullname', how='inner')

adi_df['timestamp'] = adi_df['year'].apply(lambda x: pd.to_datetime(str(x)))
adi_df.sort_index(inplace=True)

# We are missing some data: before 2011 and the year 2024.

# For year 2024 we assume to have increased more or less like inflation: 
def reconstruct_adi_y2024(df: pd.DataFrame) -> pd.DataFrame:
    # Pre allocate the nans, now we change only the values for the open municipalities in 2024
    Y2024 = pd.to_datetime("2024-01-01")
    df.loc[Y2024,: ] = np.nan
    
    # Prepare the yearly variation for year 2024
    MIN_ADI_VARIATION = 1.04
    MAX_ADI_VARIATION = 1.05

    Y2023 = pd.to_datetime("2023-01-01")
    adi_2023 = df.loc[Y2023,:].to_numpy()
    munis_2024 = df.columns[~np.isnan(adi_2023)]
    
    df.loc[Y2024, munis_2024] = df.loc[Y2023, munis_2024] * RNG.uniform(MIN_ADI_VARIATION, MAX_ADI_VARIATION, size=(len(munis_2024),))

    return df.round(1)

# For years before 2011, we need to do like we did for the business, dynamically understand which open and close 
def reconstruct_adi_until_2000(df_original: pd.DataFrame):
    """
    We reconstruct until 2000 using a constant rate around 2% 
    """
    munis_ids = df_original.columns
    n_businesses_NL = df_original.sum(axis=1, skipna=True)

    # We use the ratio between the ADI in a year and the following year
    MIN_ADI_VARIATION = 1.015
    MAX_ADI_VARIATION = 1.025

    years_to_reconstruct=pd.date_range(
        start='2000-01-01',
        end='2010-12-31',
        freq='YS'
    )
    years_to_reconstruct_plus1 = years_to_reconstruct + pd.DateOffset(years=1)
    # the only thing remaining to do is to reconstruct the municipalities that were closed before 2010
    # We will assume they had the same income of the municipality they ended up
    # We go backward splitting the businesses in the municipalities. We use the
    # same proportion of population data
    lookup_df = municipalities_df.set_index('cbs_id')
    lookup_df['begin_date'] = pd.to_datetime(lookup_df['begin_date'], errors='coerce')
    lookup_df['end_date'] = pd.to_datetime(lookup_df['end_date'], errors='coerce')
    for year, previous_year in zip(years_to_reconstruct_plus1[::-1], years_to_reconstruct[::-1]):
        # Pre-allocate the year
        df_original.loc[previous_year, :] = np.nan

        for m in df_original.columns:
            # if municipality was not open that year, leave nan
            # it closed, is the median income of the destination municipalities
            if (lookup_df.loc[m, 'begin_date'] >= year) | (pd.notna(lookup_df.loc[m, 'end_date']) & (lookup_df.loc[m, 'end_date']<year)):
                continue
            elif (pd.notna(lookup_df.loc[m, 'end_date']) & (lookup_df.loc[m, 'end_date']==year)):
                dest_munis = lookup_df.loc[m, 'destination_cbs_ids'].split(';')
                median_adi = df_original.loc[year, dest_munis].median()
                df_original.loc[previous_year, m] = median_adi * RNG.uniform(1/MAX_ADI_VARIATION, 1/MIN_ADI_VARIATION)
            else:
                df_original.loc[previous_year, m] = df_original.loc[year, m] * RNG.uniform(1/MAX_ADI_VARIATION, 1/MIN_ADI_VARIATION)

    return df_original.sort_index().round(1)


for sheet_name in ADI_COLUMNS[2:]:
    dyn_df = adi_df.pivot(index='timestamp', columns='cbs_id', values=sheet_name)

    dyn_df = reconstruct_adi_y2024(df=dyn_df)
    dyn_df = reconstruct_adi_until_2000(df_original=dyn_df)

    municipalites_dynamic_properties[sheet_name] = dyn_df

In [ ]:
# Save to Excel
municipalites_dynamic_properties.dump(path='..')